# Week 10 — Model Evaluation, Tuning, and Engineering ML Project

**ESC 113: Computer Aided Analysis for Engineering**

This notebook is designed for a **3-hour class**. Run each code cell, read the explanations, complete the guided exercises, then finish the quiz without looking at the worked examples.

## 3-hour class plan

- **0:00–0:30** Validation, cross-validation, hyperparameters
- **0:30–1:05** Pipelines and grid search
- **1:05–1:25** Comparing models fairly
- **1:25–1:35** Break
- **1:35–2:10** Concrete-strength tuning lab
- **2:10–2:40** Final project workflow and reproducibility
- **2:40–3:00** Capstone quiz / project planning

In [ ]:
!pip -q install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

concrete = fetch_ucirepo(id=165)
X = concrete.data.features.copy()
y = concrete.data.targets.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Cross-validation

Cross-validation estimates model stability using repeated training/validation splits within the training data.

In [ ]:
base_rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
scores = cross_val_score(base_rf, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
mae_scores = -scores
print("CV MAE values:", mae_scores)
print("Mean CV MAE:", mae_scores.mean())

## 2. Hyperparameter tuning

Grid search compares parameter combinations using cross-validation. The final test set should not be used to select the parameters.

In [ ]:
param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [None, 8, 16],
    "min_samples_leaf": [1, 2, 4]
}

search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
search.fit(X_train, y_train)
print(search.best_params_)
print("Best CV MAE:", -search.best_score_)

## 3. Final test evaluation

In [ ]:
best_model = search.best_estimator_
pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
print(f"Test MAE: {mae:.3f} MPa")
print(f"Test RMSE: {rmse:.3f} MPa")
print(f"Test R^2: {r2:.3f}")

## 4. Final visualization

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(y_test, pred, alpha=0.7)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, "--")
plt.xlabel("Actual Strength (MPa)")
plt.ylabel("Predicted Strength (MPa)")
plt.title("Tuned Random Forest")
plt.show()

## 5. Reproducible engineering ML checklist

A strong engineering ML project should contain:

1. Problem definition and units.
2. Data source and citation.
3. Data-quality checks.
4. Exploratory analysis.
5. Proper train/test separation.
6. Baseline model.
7. At least one stronger comparison model.
8. Metrics in engineering context.
9. Error analysis.
10. Limitations and domain-of-validity discussion.
11. Reproducible code and dependency information.
12. GitHub README and meaningful commits.

## 6. Capstone exercises

1. Compare the tuned random forest with an untuned baseline.
2. Save the comparison to a DataFrame.
3. Plot the absolute test error distribution.
4. Identify the five largest prediction errors and inspect their feature values.
5. Write a paragraph explaining whether the model should replace laboratory compression testing.
6. State the model's likely domain of validity and two reasons predictions may fail in practice.

## 7. Project planning activity

Create a short plan for your own Project 2 repository:

- research/engineering question
- dataset source
- target variable
- input features
- baseline model
- comparison models
- metrics
- plots
- expected report sections
- GitHub file structure

## 8. Final quiz

1. Why should the test set be used only at the end?
2. What is cross-validation?
3. What is a hyperparameter?
4. What does GridSearchCV do?
5. Define overfitting and underfitting.
6. Why should model performance be reported in engineering units?
7. Why must limitations be discussed in an engineering ML report?
8. Name three files that should be present in the final GitHub project repository.